In [1]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix

ratings  = pd.read_parquet('D:/documents/Smart_Library/data/processed/ratings_clean.parquet')
explicit = ratings[ratings['event_type'] == 'explicit_rating'].copy()

print(f"Training on {len(explicit)} explicit ratings")
print(f"Traing userss: {explicit['User-ID'].nunique()}")
print(f"Unique books: {explicit['ISBN'].nunique()}")


Training on 433671 explicit ratings
Traing userss: 77805
Unique books: 185973


In [2]:
print(f"Total row loaded:{len(ratings)}")
print(ratings.columns.tolist())
print(ratings['event_type'].value_counts())

Total row loaded:1149780
['User-ID', 'ISBN', 'Book-Rating', 'rating_normalized', 'event_type']
event_type
implicit_interaction    716109
explicit_rating         433671
Name: count, dtype: int64


In [3]:
explicit['user_idx'] = explicit['User-ID'].astype('category').cat.codes
explicit['book_idx'] = explicit['ISBN'].astype('category').cat.codes

user_ids = explicit['User-ID'].astype('category').cat.categories
book_isbns = explicit['ISBN'].astype('category').cat.categories

matrix = csr_matrix(
    (explicit['rating_normalized'], (explicit['user_idx'], explicit['book_idx']))
)


print(f"Matrix shape: {matrix.shape}")
print(f"Non-zerp entries: {matrix.nnz}")
print(f"Sparsity: {(1- matrix.nnz/(matrix.shape[0] * matrix.shape[1])) *100:.4f}% empty")

Matrix shape: (77805, 185973)
Non-zerp entries: 433671
Sparsity: 99.9970% empty


In [4]:
from implicit.als import AlternatingLeastSquares

model = AlternatingLeastSquares(
    factors = 64,
    regularization=0.1,
    iterations=20,
    random_state=42
)

model.fit(matrix)
print("Training Complete")

d:\documents\Smart_Library\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\documents\Smart_Library\venv\Lib\site-packages\implicit\cpu\als.py:96: RuntimeWarning: OpenBLAS is configured to use 8 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100%|██████████| 20/20 [00:17<00:00,  1.12it/s]

Training Complete


In [2]:
books = pd.read_parquet('D:/documents/Smart_Library/data/processed/books_with_mood.parquet')


In [6]:
test_user_idx = 0
recommended = model.recommend(test_user_idx, matrix[test_user_idx],N=5)

for book_idx, score in  zip(recommended[0], recommended[1]):
    isbn = book_isbns[book_idx]
    title = books[books['isbn13'] == isbn]['title'].values
    title = title[0] if len(title) > 0 else "(title not found)"
    print(f"{title} - score {score:.3f}")

(title not found) - score 0.006
(title not found) - score 0.005
(title not found) - score 0.005
(title not found) - score 0.004
(title not found) - score 0.003


In [7]:
print("sample from book_isbn (rating file):", book_isbns[:5].tolist())
print("Sample from books['isbn13] (catalog)", books['isbn13'].head(5).tolist())

sample from book_isbn (rating file): [' 0330299891', ' 0375404120', ' 9022906116', '#6612432', "'9607092910'"]
Sample from books['isbn13] (catalog) ['9780452264465', '9780440237228', '9780515128550', '9780385472951', '9780380789016']


Full audit of ratingss

In [8]:
print(f"Rows: {len(ratings)}")
print(ratings.isna().sum())

print("\n User-ID:")
print(ratings['User-ID'].describe())

print("\n ISBN:")
print(ratings['ISBN'].astype(str).str.len().value_counts().sort_index())
print("Junk chars:", ratings['ISBN'].astype(str).str.contains(r'[^0-9Xx\s]').sum())

print("\n Book Rating:")
print(ratings['Book-Rating'].describe())

print("\n Duplicates")
print("Same user+ISBN more than once:", ratings.duplicated(subset=['User-ID','ISBN'], keep= False).sum())
print("Fully identical rows", ratings.duplicated().sum())

Rows: 1149780
User-ID                   0
ISBN                      0
Book-Rating               0
rating_normalized    716109
event_type                0
dtype: int64

 User-ID:


count    1.149780e+06
mean     1.403864e+05
std      8.056228e+04
min      2.000000e+00
25%      7.034500e+04
50%      1.410100e+05
75%      2.110280e+05
max      2.788540e+05
Name: User-ID, dtype: float64

 ISBN:
ISBN
8         677
9        5139
10    1139356
11       2153
12        704
13       1738
14         12
17          1
Name: count, dtype: int64
Junk chars: 2397

 Book Rating:
count    1.149780e+06
mean     2.866950e+00
std      3.854184e+00
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      7.000000e+00
max      1.000000e+01
Name: Book-Rating, dtype: float64

 Duplicates
Same user+ISBN more than once: 0
Fully identical rows 0


In [13]:
import re

def isbn10_to_isbn13(isbn10):
    isbn10 = str(isbn10).strip().replace('-', '')
    if len(isbn10)!=10:
        return None
    core = '978' + isbn10[:9]
    total= sum(int(d)*(1 if i%2 == 0 else 3) for i,d in enumerate(core) if d.isdigit())
    check = (10 - (total%10)) % 10
    return core+ str(check)


def to_isbn13(isbn):
    isbn  = str(isbn).strip().replace('-','')
    if len(isbn) == 13:
        result = isbn
    elif len(isbn) == 10:
        result =  isbn10_to_isbn13(isbn)
    else:
        return None
    if result and result.lstrip('0') == '':
        return None
    return result


def clean_isbn(isbn):
    if pd.isna(isbn):
        return None
    isbn = str(isbn).strip().strip("'\"")
    isbn =  re.sub(r'[^0-9Xx]', '', isbn)
    return isbn if isbn else None

ratings['isbn_cleaned'] = ratings['ISBN'].apply(clean_isbn)
ratings['isbn13'] = ratings['isbn_cleaned'].apply(to_isbn13)

print(f"Converted: {ratings['isbn13'].notna().sum()}")
print(f"Dropped (unrecovered): {ratings['isbn13'].isna().sum()}")


Converted: 1140510
Dropped (unrecovered): 9270


Rebuild the corrected isbn13

In [7]:
explicit =  ratings[(ratings['event_type'] == 'explicit_rating') & (ratings['isbn13'].notna())].copy()

print(f"Explicit ratings with valid isbn13: {len(explicit)}")

explicit['user_idx'] = explicit['User-ID'].astype('category').cat.codes
explicit['book_idx'] = explicit['isbn13'].astype('category').cat.codes

user_ids = explicit['User-ID'].astype('category').cat.categories
book_isbns = explicit['isbn13'].astype('category').cat.categories

matrix = csr_matrix(
    (explicit['rating_normalized'], (explicit['user_idx'], explicit['book_idx']))
)


print(f"Matrix shape: {matrix.shape}")
print(f"Non-zerp entries: {matrix.nnz}")
print(f"Sparsity: {(1- matrix.nnz/(matrix.shape[0] * matrix.shape[1])) *100:.4f}% empty")

Explicit ratings with valid isbn13: 429532
Matrix shape: (76870, 181599)
Non-zerp entries: 429485
Sparsity: 99.9969% empty


In [8]:
from implicit.als import AlternatingLeastSquares

model = AlternatingLeastSquares(
    factors = 64,
    regularization=0.1,
    iterations=20,
    random_state=42
)

model.fit(matrix)
print("Training Complete")

100%|██████████| 20/20 [00:17<00:00,  1.14it/s]

Training Complete


In [12]:
books = pd.read_parquet('D:/documents/Smart_Library/data/processed/books_with_mood.parquet')


In [9]:
test_user_idx = 0
recommended = model.recommend(test_user_idx, matrix[test_user_idx],N=5)

for book_idx, score in  zip(recommended[0], recommended[1]):
    isbn = book_isbns[book_idx]
    title = books[books['isbn13'] == isbn]['title'].values
    title = title[0] if len(title) > 0 else "(title not found)"
    print(f"{title} - score {score:.3f}")

The Firm - score 0.009
The Red Tent (Bestselling Backlist) - score 0.004
Jackdaws - score 0.004
Snow Falling on Cedars - score 0.003
Girl with a Pearl Earring - score 0.003


In [10]:
ratings_isbns = set(book_isbns)
catalog_isbns = set(books['isbn13'].dropna())
overlap = ratings_isbns & catalog_isbns

print(f"Unique ISBNS in ratings: {len(ratings_isbns)}")
print(f"Unique ISBNS in catalog: {len(catalog_isbns)}")
print(f"Overlap: {len(overlap)} ({len(overlap)/len(ratings_isbns)*100:.1f}%")

for book_idx, score in zip(recommended[0], recommended[1]):
    isbn  = book_isbns[book_idx]
    print(repr(isbn), "in catalog: ", isbn in catalog_isbns)

Unique ISBNS in ratings: 181599
Unique ISBNS in catalog: 278021
Overlap: 149575 (82.4%
'9780440211457' in catalog:  True
'9780312195519' in catalog:  True
'9780451207524' in catalog:  True
'9780679764021' in catalog:  True
'9780452282155' in catalog:  True


In [11]:
print(explicit['isbn13'].head(3).tolist())
print(book_isbns[:3].tolist())

['9780155061224', '9780521656153', '9780521795029']
['0000000000000', '0000000000090', '0000000016964']


In [14]:
zero_isbns = explicit[explicit['isbn13'].str.startswith('000')]
print(f"Rows with suspicious all/near-zero inputs isbn13: {len(zero_isbns)}")
print(zero_isbns[['ISBN', 'isbn13']].head(5))

Rows with suspicious all/near-zero inputs isbn13: 14
                 ISBN         isbn13
65999   0000000000090  0000000000090
66480   0000000016964  0000000016964
86679   0000000048118  0000000048118
140207  0000000000000  0000000000000
240855  0000000030670  0000000030670
